# Receipt NER v2: LayoutLMv3 with real visual features

The first version of this model (plain LayoutLM - just text and rough word
position, no image) kept getting TOTAL wrong in a specific way: 73.8% of
its errors were picking a real number from somewhere else on the receipt -
a line item, the subtotal, tax - not nonsense. That's exactly the kind of
mistake a person avoids just by looking at the receipt, since the real
total is usually bigger, bolder, or boxed. v1 never sees the image at all,
so it has no way to notice that. This notebook tries LayoutLMv3, which
does look at the actual pixels, to see if that closes the gap.

## Checking this was doable before writing any training code

LayoutLMv3 was used instead of v2 because v2 needs Detectron2 for its
visual backbone, and Detectron2 has no official Windows build - it's a
known headache involving manual C++ build tools and hand-compiled CUDA
extensions. v3 avoids all that with a simpler ViT-style patch embedding
that's already built into the `transformers` library, so nothing extra
had to be installed.

GPU memory was also a concern, since this machine only has a GTX 1650 with
4GB of VRAM. One real training step (batch size 1, fp16, 512 tokens, a
224x224 image) peaked at 2.37GB, which leaves comfortable room - good,
because `LayoutLMv3ForTokenClassification` doesn't support gradient
checkpointing at all (it just raises an error if you try), so the
memory-saving trick v1 depended on wasn't an option here.

The TFLite conversion path was also tested against the real trained model
before writing the conversion code in Cell 8, rather than assuming it
would just work. Converting PyTorch to TensorFlow gave 100% matching
predictions. Full INT8 quantization failed (a TRANSPOSE op inside the
visual patch embedding has no integer version), but falling back to
dynamic-range INT8 worked fine and produced a ~146MB file - see Cell 8 for
the exact numbers.

## What changed from `receipt_ner_gpu_training.ipynb`

Every training example now also loads its receipt image and produces a
`pixel_values` tensor (224x224x3, via `LayoutLMv3ImageProcessor`).
Everything else about finding and labeling words stays exactly the same
tested pipeline from v1, reused as-is, with one small addition:
`build_example` now also returns the image path.

`LayoutLMv3TokenizerFast` also matches words to their boxes and labels on
its own when you pass `boxes=`/`word_labels=` directly, so the manual
alignment loop v1 needed isn't necessary here.

Everything else - learning rate, schedule, class weighting, random seed,
number of epochs - was kept exactly the same as v1 on purpose, so if
accuracy changes, it's because of the architecture and nothing else.

## Update: TFLite conversion added (Cells 8+)

This notebook originally stopped after training and evaluation, before it
was clear whether the accuracy gain would be worth the extra work of
deploying it. It was: the test set came back with an overall F1 of 0.826,
and TOTAL specifically went from 0.693 raw to 0.913 with the largest-box
fix applied - a big jump over v1 on every field (Company went from about
14% to 93.6%, Date from 62% to 95.4%, Address from 10% to 76.8%). Cells 8
onward add the PyTorch -> TensorFlow -> TFLite conversion, following the
same steps as v1 but handling the extra `pixel_values` input. Still
missing: the actual Flutter-side code to preprocess a photo into
`pixel_values` on-device - that's new work this project didn't need
before, saved for later.

## Cell 1 - Dependencies & environment

Same as `receipt_ner_gpu_training.ipynb` - no new installs needed,
LayoutLMv3 comes from the same `transformers` package.

In [ ]:
# Pins: transformers<5 (v5 dropped the TF* classes Cell 4 needs for PT->TF conversion); setuptools_scm<8 before seqeval (its old build otherwise crashes against newer setuptools_scm); torch from PyTorch's own CUDA index since plain pip can silently resolve a CPU-only build even with a GPU present (cu132 matches CUDA 13.2 - lower it if your driver's older, check `nvidia-smi`).
!pip install -q "setuptools_scm<8"
!pip install -q "transformers<5" datasets accelerate evaluate seqeval tf-keras tensorflow pillow tqdm matplotlib
!pip install -q torch --index-url https://download.pytorch.org/whl/cu132

import torch
import tensorflow as tf

print(f"PyTorch version: {torch.__version__}")
print(f"TensorFlow version: {tf.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")
else:
    print("WARNING: no GPU detected. Training will fall back to CPU, which will be very slow for a "
          "BERT-base-sized model over 10 epochs. Check `nvidia-smi` at a terminal to confirm the "
          "driver is visible, and that `pip install torch` above actually resolved a CUDA build "
          "(torch.__version__ should NOT end in '+cpu').")


## Cell 2 - Load and label the SROIE dataset

Same data-loading/labeling pipeline as `receipt_ner_gpu_training.ipynb` -
copied over as-is (already validated: field coverage, spot checks, a full
label-quality audit), with one patch: `build_example` also returns
`image_path` now, needed to load `pixel_values`.

In [ ]:
import json
import re
import difflib
from pathlib import Path
from PIL import Image
from tqdm import tqdm

# --- Locate the dataset -----------------------------------------------
_CANDIDATE_ROOTS = [
    Path("c:/Users/User/Desktop/expenseTracker/ai/data/raw/SROIE2019"),  # local, absolute (this machine)
    Path("data/raw/SROIE2019"),                          # local, cwd == ai/
    Path("ai/data/raw/SROIE2019"),                       # local, cwd == repo root
    Path("/kaggle/input/sroie-datasetv2/SROIE2019"),     # Kaggle, dataset added as input
    Path("/kaggle/input/sroie-dataset/SROIE2019"),        # alternate Kaggle mirror naming
    Path("./SROIE2019"),                                  # Colab, manually placed
    Path("/content/SROIE2019"),                           # Colab, uploaded to /content
]
DATASET_ROOT = next((p for p in _CANDIDATE_ROOTS if p.exists()), None)
if DATASET_ROOT is None:
    raise FileNotFoundError(
        "SROIE2019 not found in any of: " + ", ".join(str(p) for p in _CANDIDATE_ROOTS) +
        ". Check the VSCode Jupyter kernel's working directory (Cell > Execute in a terminal at "
        "the repo root or ai/ to confirm), or add the correct path to _CANDIDATE_ROOTS."
    )
print(f"Using dataset at: {DATASET_ROOT}")

# --- LayoutLM checkpoint ---------------------------------------------------
# Use the official HF Hub checkpoint, not the local copy at ai/data/raw/SROIE2019/layoutlm-base-uncased/ - that one uses Microsoft's original bert.*/cls.* key names, which transformers silently fails to map onto layoutlm.*/classifier.* (no error, it just reinitializes every weight from scratch - confirmed all ~200 params showed up as newly initialized). The Hub checkpoint uses the right key names (only classifier.weight/bias get reinitialized, as expected).
MODEL_NAME = "microsoft/layoutlm-base-uncased"
print(f"Using LayoutLM checkpoint: {MODEL_NAME} (downloads from the HF Hub on first run, then cached)")

# --- BIO label scheme (4 fields x B/I, plus O) ---------------------------
FIELDS = ["COMPANY", "DATE", "ADDRESS", "TOTAL"]
LABELS = ["O"] + [f"{prefix}-{field}" for field in FIELDS for prefix in ("B", "I")]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for i, label in enumerate(LABELS)}
IGNORE_LABEL_ID = -100  # HF convention: skip this token when computing loss/metrics

LINE_MATCH_RATIO = 0.8   # line-level fuzzy threshold (assign_line_label)
WORD_MATCH_RATIO = 0.7   # word-level fuzzy threshold (second-pass filter)
ADDRESS_PARTIAL_MATCH = 0.5


def read_bbox_lines(path):
    """Parses one SROIE box/*.txt file: each physical OCR line is
    'x0,y0,x1,y1,x2,y2,x3,y3,text' (a quadrilateral + text) - text itself
    can contain commas, so this splits on the first 8 commas only.
    Returns a list of {x0,y0,x2,y2,text} dicts, one per physical line
    (top-left / bottom-right corners only, matching the reference
    notebook's own simplification of the quadrilateral to a rectangle)."""
    lines = []
    for raw_line in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        if not raw_line:
            continue
        parts = raw_line.split(",", 8)
        if len(parts) < 9:
            continue  # malformed line (rare OCR/annotation artifact) - skip rather than guess
        try:
            x0, y0, x2, y2 = int(parts[0]), int(parts[1]), int(parts[4]), int(parts[5])
        except ValueError:
            continue
        lines.append({"x0": x0, "y0": y0, "x2": x2, "y2": y2, "text": parts[8]})
    return lines


def read_entities(path):
    return json.loads(path.read_text(encoding="utf-8", errors="ignore"))


def norm_match(text):
    """Strips to alphanumeric-lowercase so fuzzy comparisons are symmetric -
    a short word like '6,' vs '6' can otherwise fall below the match
    threshold on punctuation alone (ratio 0.667 vs. a 0.7 cutoff), even
    though the content is identical."""
    return re.sub(r"[^a-z0-9]", "", text.lower())


In [ ]:
# --- Step 1: line-level label assignment (reused from the reference notebook) ---
def assign_line_label(line_text, entities):
    """Fuzzy-matches every word in one OCR line against each field's
    ground-truth entity value. ADDRESS only needs half its words to match
    (addresses commonly span multiple OCR lines, so no single line has to
    account for the whole address); every other field needs either the
    whole line or the whole entity value accounted for."""
    line_words = line_text.replace(",", "").strip().split()
    if not line_words:
        return "O"

    for field in FIELDS:
        entity_value = entities.get(field.lower(), "").replace(",", "").strip()
        entity_words = entity_value.split()
        if not entity_words:
            continue

        matches = 0
        for word in line_words:
            if any(difflib.SequenceMatcher(None, norm_match(word), norm_match(ew)).ratio() > LINE_MATCH_RATIO for ew in entity_words):
                matches += 1
            if (field == "ADDRESS" and matches / len(line_words) >= ADDRESS_PARTIAL_MATCH) or \
               (field != "ADDRESS" and matches == len(line_words)) or \
               matches == len(entity_words):
                return field
    return "O"


def assign_labels(lines, entities):
    """Runs assign_line_label top-to-bottom over every line, then applies
    two corrections: (1) suppress a COMPANY/ADDRESS match found after a
    DATE/TOTAL line has already been seen (footer text re-mentioning the
    merchant), and (2) for TOTAL/DATE specifically, keep only the single
    candidate line with the largest bounding box (width + height) - a
    receipt's real total/date is conventionally the most visually
    prominent instance, and duplicated numbers elsewhere shouldn't win."""
    max_score = {"TOTAL": (-1, -1), "DATE": (-1, -1)}  # (score, line_index)
    already_labeled = {"TOTAL": False, "DATE": False, "ADDRESS": False, "COMPANY": False, "O": False}

    labels = []
    for i, line in enumerate(lines):
        label = assign_line_label(line["text"], entities)
        already_labeled[label] = True

        if (label == "ADDRESS" and already_labeled["TOTAL"]) or \
           (label == "COMPANY" and (already_labeled["DATE"] or already_labeled["TOTAL"])):
            label = "O"

        if label in ("TOTAL", "DATE"):
            score = (line["x2"] - line["x0"]) + (line["y2"] - line["y0"])
            if score > max_score[label][0]:
                max_score[label] = (score, i)
            label = "O"  # provisional - only the single winning line gets it back below

        labels.append(label)

    if max_score["DATE"][1] != -1:
        labels[max_score["DATE"][1]] = "DATE"
    if max_score["TOTAL"][1] != -1:
        labels[max_score["TOTAL"][1]] = "TOTAL"
    return labels


In [ ]:
# --- Step 2: per-word bbox splitting (reused from the reference notebook) ---
def split_line_bbox(line):
    """A box file gives one bounding box per LINE. Approximates a
    per-word box by splitting the line's box proportionally to each
    word's character-length share of the line's total character length -
    not pixel-accurate, but the same approximation the reference notebook
    uses, and good enough for LayoutLM's coarse (0-1000 bucketed) position
    embeddings."""
    line_str = line["text"]
    words = [w for w in line_str.split(" ") if len(w) >= 1]
    if not words or len(line_str) == 0:
        return []

    x0, y0, x2_line, y2 = line["x0"], line["y0"], line["x2"], line["y2"]
    bbox_width = x2_line - x0

    result = []
    cur_x0 = x0
    for word in words:
        cur_x2 = cur_x0 + int(bbox_width * len(word) / len(line_str))
        result.append({"text": word, "x0": cur_x0, "y0": y0, "x2": cur_x2, "y2": y2})
        cur_x0 = cur_x2 + 5  # small gap between words, matching the reference notebook
    return result


# --- Step 3 + 4: per-word fuzzy filter, then fold into BIO spans ---------
def resolve_word_labels(exploded_words):
    """exploded_words: list of (word_dict, candidate_field) pairs, where
    candidate_field is the parent line's field label ("O" for most words).
    For words whose parent line carries a real field label, fuzzy-checks
    the individual word against that field's entity tokens (a line
    matching at the line level doesn't mean every word in it belongs to
    the field - e.g. 'TOTAL:' in a line 'TOTAL: RM20.80' shouldn't be
    tagged). Then folds consecutive same-field words into BIO spans."""
    resolved = []
    for word, candidate_field in exploded_words:
        if candidate_field == "O":
            resolved.append("O")
            continue
        entity_value = _CURRENT_ENTITIES.get(candidate_field.lower(), "").replace(",", "").strip()
        entity_words = entity_value.split()
        matched = bool(entity_words) and any(
            difflib.SequenceMatcher(None, norm_match(word["text"]), norm_match(ew)).ratio() > WORD_MATCH_RATIO for ew in entity_words
        )
        resolved.append(candidate_field if matched else "O")

    tags, previous_field = [], None
    for field in resolved:
        if field == "O":
            tags.append("O")
            previous_field = None
        elif field == previous_field:
            tags.append(f"I-{field}")
        else:
            tags.append(f"B-{field}")
            previous_field = field
    return tags


In [ ]:
# --- Step 5: assemble one receipt (words + normalized 2D boxes + BIO tags) ---
_CURRENT_ENTITIES = {}  # set per-receipt inside build_example; read by resolve_word_labels


def normalize_bbox(word, page_width, page_height):
    """Scales a word's bounding box to LayoutLM's expected 0-1000 range
    relative to the receipt image's own dimensions, clamped defensively in
    case of any off-by-a-pixel rounding at the image edges."""
    def scale(value, extent):
        return max(0, min(1000, int(value * 1000 / extent))) if extent > 0 else 0

    return [
        scale(word["x0"], page_width),
        scale(word["y0"], page_height),
        scale(word["x2"], page_width),
        scale(word["y2"], page_height),
    ]


def build_example(box_path, entities_path, img_path):
    global _CURRENT_ENTITIES

    lines = read_bbox_lines(box_path)
    if not lines:
        return None
    try:
        entities = read_entities(entities_path)
    except json.JSONDecodeError:
        return None
    try:
        with Image.open(img_path) as img:
            page_width, page_height = img.size
    except (FileNotFoundError, OSError):
        return None

    _CURRENT_ENTITIES = entities
    line_labels = assign_labels(lines, entities)

    exploded = []
    for line, label in zip(lines, line_labels):
        for word in split_line_bbox(line):
            exploded.append((word, label))
    if not exploded:
        return None

    tags = resolve_word_labels(exploded)
    words = [word["text"] for word, _ in exploded]
    boxes = [normalize_bbox(word, page_width, page_height) for word, _ in exploded]

    return {
        "words": words,
        "boxes": boxes,
        "ner_tags": tags,
        "image_path": str(img_path),
        "_total_found": any(tag == "B-TOTAL" for tag in tags),
    }


def load_split(split_dir):
    examples, unmatched = [], 0
    box_dir, entities_dir, img_dir = split_dir / "box", split_dir / "entities", split_dir / "img"
    box_paths = sorted(box_dir.glob("*.txt"))

    for box_path in tqdm(box_paths, desc=f"Loading {split_dir.name}", unit="receipt"):
        entities_path = entities_dir / box_path.name
        img_path = img_dir / (box_path.stem + ".jpg")
        if not entities_path.exists() or not img_path.exists():
            continue
        example = build_example(box_path, entities_path, img_path)
        if example is None:
            continue
        if not example.pop("_total_found"):
            unmatched += 1
            continue  # no reliable TOTAL ground truth - drop rather than mislabel every word O
        examples.append(example)

    print(f"  {split_dir.name}: {len(examples)} labeled, {unmatched} skipped (total not found)")
    return examples


print("Loading SROIE2019...")
train_examples = load_split(DATASET_ROOT / "train")
test_examples = load_split(DATASET_ROOT / "test")

for split_name, examples in [("train", train_examples), ("test", test_examples)]:
    coverage = {
        field: sum(1 for ex in examples if f"B-{field}" in ex["ner_tags"])
        for field in FIELDS
    }
    print(f"  {split_name} field coverage: {coverage} out of {len(examples)}")


## Cell 3 - Tokenizer, image processor, and a FIXED validation/test split

`LayoutLMv3TokenizerFast(..., boxes=..., word_labels=...)` handles
word-to-subword alignment internally - checked it against a real example
and it matches v1's `-100`-for-continuation convention. `apply_ocr=False`
on the image processor matters: without it, the processor runs its own
OCR via pytesseract and throws away the words/boxes this pipeline already
built.

In [ ]:
from transformers import LayoutLMv3TokenizerFast, LayoutLMv3ImageProcessor
from datasets import Dataset
from PIL import Image as PILImage

MODEL_NAME = "microsoft/layoutlmv3-base"
MAX_LENGTH = 512

tokenizer = LayoutLMv3TokenizerFast.from_pretrained(MODEL_NAME)
image_processor = LayoutLMv3ImageProcessor(apply_ocr=False)


def tokenize_and_align(batch):
    """boxes=/word_labels= handle subword alignment internally - no
    manual word_ids() loop needed here, unlike v1. Still padding=False for
    the text/box/label fields (dynamic per-batch padding via the collator
    below, same reasoning as v1 - most receipts are well under the 512
    token ceiling). pixel_values is a fixed 224x224x3 regardless of
    receipt content, so it never needs padding.
    """
    label_ids = [[LABEL2ID[tag] for tag in tags] for tags in batch["ner_tags"]]
    tokenized = tokenizer(
        batch["words"],
        boxes=batch["boxes"],
        word_labels=label_ids,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

    pixel_values = []
    for image_path in batch["image_path"]:
        with PILImage.open(image_path) as img:
            pixel_values.append(image_processor(img.convert("RGB"), return_tensors="np")["pixel_values"][0])
    tokenized["pixel_values"] = pixel_values
    return tokenized


# 90/10 train/validation split (test_examples stays untouched until final eval); same seed(42) shuffle as v1 so the two are directly comparable.
import random
random.seed(42)
shuffled = train_examples[:]
random.shuffle(shuffled)
val_size = max(1, len(shuffled) // 10)
val_examples, train_only_examples = shuffled[:val_size], shuffled[val_size:]

raw_datasets = {
    "train": Dataset.from_list(train_only_examples),
    "validation": Dataset.from_list(val_examples),
    "test": Dataset.from_list(test_examples),
}

tokenized_datasets = {
    split: ds.map(tokenize_and_align, batched=True, remove_columns=ds.column_names)
    for split, ds in raw_datasets.items()
}

print(f"train: {len(tokenized_datasets['train'])}, "
      f"validation: {len(tokenized_datasets['validation'])}, "
      f"test: {len(tokenized_datasets['test'])}")


## Cell 4 - Class weighting and the weighted-loss trainer

Same as `receipt_ner_gpu_training.ipynb` - same `MAX_CLASS_WEIGHT=5.0` for
the same reason (precision/recall imbalance measured there).

In [ ]:
import torch
import numpy as np
import torch.nn as nn
from collections import Counter
from transformers import (
    LayoutLMv3ForTokenClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)

MAX_CLASS_WEIGHT = 5.0


def compute_class_weights(tokenized_train):
    counts = Counter()
    for labels in tokenized_train["labels"]:
        counts.update(l for l in labels if l != IGNORE_LABEL_ID)
    total = sum(counts.values())
    weights = np.ones(len(LABELS), dtype=np.float32)
    for label_id in range(len(LABELS)):
        count = counts.get(label_id, 1)  # avoid /0 for a label that never occurs
        weights[label_id] = min(total / (len(LABELS) * count), MAX_CLASS_WEIGHT)
    return torch.tensor(weights, dtype=torch.float32)


class_weights = compute_class_weights(tokenized_datasets["train"])
print("Class weights:", {label: round(float(w), 2) for label, w in zip(LABELS, class_weights)})


class WeightedLossTrainer(Trainer):
    """Overrides Trainer's default (unweighted) loss with a class-weighted
    cross-entropy - HF's Trainer has no built-in per-class weighting for
    token classification, so this is the standard way to add it."""

    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device), ignore_index=IGNORE_LABEL_ID
        )
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


## Cell 5 - Collator and metrics

Same dynamic per-batch padding as v1's `DynamicPaddingCollator`, plus
stacking `pixel_values` (always a fixed 224x224x3, so no padding needed
there). Same `compute_metrics` as v1 - per-field seqeval F1, not just
overall.

In [ ]:
import evaluate

seqeval_metric = evaluate.load("seqeval")


class DynamicPaddingCollator:
    def __init__(self, pad_token_id, label_pad_id=IGNORE_LABEL_ID):
        self.pad_token_id = pad_token_id
        self.label_pad_id = label_pad_id

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        batch = {"input_ids": [], "attention_mask": [], "labels": [], "bbox": []}

        for f in features:
            pad_len = max_len - len(f["input_ids"])
            batch["input_ids"].append(f["input_ids"] + [self.pad_token_id] * pad_len)
            batch["attention_mask"].append(f["attention_mask"] + [0] * pad_len)
            batch["labels"].append(f["labels"] + [self.label_pad_id] * pad_len)
            batch["bbox"].append(f["bbox"] + [[0, 0, 0, 0]] * pad_len)

        result = {key: torch.tensor(value, dtype=torch.long) for key, value in batch.items()}
        result["pixel_values"] = torch.tensor(np.array([f["pixel_values"] for f in features]), dtype=torch.float32)
        return result


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [ID2LABEL[p] for p, l in zip(pred, label) if l != IGNORE_LABEL_ID]
        for pred, label in zip(predictions, labels)
    ]
    true_labels = [
        [ID2LABEL[l] for p, l in zip(pred, label) if l != IGNORE_LABEL_ID]
        for pred, label in zip(predictions, labels)
    ]

    results = seqeval_metric.compute(predictions=true_predictions, references=true_labels)
    metrics = {
        "overall_f1": results["overall_f1"],
        "overall_precision": results["overall_precision"],
        "overall_recall": results["overall_recall"],
    }
    for field in FIELDS:
        metrics[f"{field.lower()}_f1"] = results.get(field, {}).get("f1", 0.0)
    return metrics


## Cell 6 - Model, training arguments, and training

No `gradient_checkpointing` (not supported by this model class, see the
opening cell). Batch size stays at 1 despite the 2.37GB/4GB headroom -
keeping it the same as v1's proven-safe setting rather than changing two
things at once.

In [ ]:
model = LayoutLMv3ForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,  # replaces the pretrained checkpoint's own classifier head with a fresh 9-label one
)


In [ ]:
# set_seed() before model creation, not after - TrainingArguments' own seed only kicks in once Trainer.__init__ runs, by which point the classifier head's already randomly initialized.
set_seed(42)

EPOCHS = 25

training_args = TrainingArguments(
    output_dir="./training_checkpoints_v3",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    learning_rate=3e-5,
    lr_scheduler_type="linear",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,      # effective batch size 4, matching v1
    per_device_eval_batch_size=2,
    # gradient_checkpointing intentionally omitted - LayoutLMv3ForTokenClassification
    # does not support it (confirmed: raises ValueError if set).
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    fp16=True,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
    load_best_model_at_end=True,
    metric_for_best_model="total_f1",   # NOT blended/overall F1 - same reasoning as v1
    greater_is_better=True,
    logging_steps=10,
    report_to="none",
    seed=42,
)

trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=DynamicPaddingCollator(pad_token_id=tokenizer.pad_token_id),
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)


In [ ]:
import os
import glob

# Resumes automatically if a previous run was interrupted, same as v1.
_has_checkpoint = bool(glob.glob(os.path.join(training_args.output_dir, "checkpoint-*")))
if _has_checkpoint:
    print(f"Resuming from the latest checkpoint in: {training_args.output_dir}")

trainer.train(resume_from_checkpoint=_has_checkpoint or None)

# Final eval on the held-out test set, untouched until now - compare directly against v1's test_total_f1=0.2020 (0.39 with the largest-box fix, see Cell 7).
test_results = trainer.evaluate(tokenized_datasets["test"])
print("\nTest set results:")
for key, value in test_results.items():
    if key.startswith("eval_"):
        print(f"  {key[5:]}: {value:.4f}" if isinstance(value, float) else f"  {key[5:]}: {value}")

SAVE_DIR = "./saved_pytorch_model_v3"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"\nSaved PyTorch model + tokenizer to {SAVE_DIR}")


## Cell 7 - Apply the same largest-bounding-box post-processing fix

The fix that took v1's TOTAL accuracy from 17.4% to 39.0% - picking the
largest-bounding-box candidate instead of just the first one - doesn't
depend on the architecture, so it's worth applying here too. Comparing
v3-with-fix against v1-with-fix is the fair test; v3-raw vs. v1-with-fix
wouldn't be.

In [ ]:
def predict_word_tags(words, boxes, pixel_values):
    encoding = tokenizer(words, boxes=boxes, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    encoding = {k: v.to(trainer.model.device) for k, v in encoding.items()}
    encoding["pixel_values"] = torch.tensor(np.array([pixel_values]), dtype=torch.float32).to(trainer.model.device)

    with torch.no_grad():
        logits = trainer.model(**encoding).logits
    predictions = logits.argmax(-1)[0].cpu().tolist()

    word_ids = tokenizer(words, boxes=boxes, truncation=True, max_length=MAX_LENGTH).word_ids()
    word_tags = ["O"] * len(words)
    seen_words = set()
    for token_index, word_id in enumerate(word_ids):
        if word_id is None or word_id in seen_words:
            continue
        seen_words.add(word_id)
        word_tags[word_id] = ID2LABEL[predictions[token_index]]
    return word_tags


def line_area(box):
    x0, y0, x2, y2 = box
    return max(0, x2 - x0) + max(0, y2 - y0)


def extract_spans(words, boxes, word_tags, field):
    spans = []
    i = 0
    while i < len(words):
        if word_tags[i] == f"B-{field}":
            span_words, span_boxes = [words[i]], [boxes[i]]
            j = i + 1
            while j < len(words) and word_tags[j] == f"I-{field}":
                span_words.append(words[j])
                span_boxes.append(boxes[j])
                j += 1
            x0 = min(b[0] for b in span_boxes); y0 = min(b[1] for b in span_boxes)
            x2 = max(b[2] for b in span_boxes); y2 = max(b[3] for b in span_boxes)
            spans.append((" ".join(span_words), (x0, y0, x2, y2)))
            i = j
        else:
            i += 1
    return spans


def group_fields(words, boxes, word_tags):
    result = {field: "" for field in FIELDS}
    for field in FIELDS:
        spans = extract_spans(words, boxes, word_tags, field)
        if not spans:
            continue
        result[field] = max(spans, key=lambda s: line_area(s[1]))[0] if field in ("TOTAL", "DATE") else spans[0][0]
    return result


import re
from collections import Counter


def normalize_amount(text):
    return re.sub(r"[^0-9.]", "", text)


total_categories = Counter()
for example, tokenized in zip(test_examples, tokenized_datasets["test"]):
    word_tags = predict_word_tags(example["words"], example["boxes"], tokenized["pixel_values"])
    predicted = group_fields(example["words"], example["boxes"], word_tags)
    gt_total = " ".join(w for w, t in zip(example["words"], example["ner_tags"]) if "TOTAL" in t)

    gt_norm, pred_norm = normalize_amount(gt_total), normalize_amount(predicted["TOTAL"])
    if pred_norm == gt_norm:
        total_categories["correct"] += 1
    elif not pred_norm:
        total_categories["missed"] += 1
    else:
        other_numbers = {normalize_amount(w) for w in example["words"] if normalize_amount(w)}
        other_numbers.discard(gt_norm)
        total_categories["decoy" if pred_norm in other_numbers else "other"] += 1

print("v3 TOTAL results (with the largest-box fix applied) - compare directly against v1's 39.0%:")
for category, count in total_categories.most_common():
    print(f"  {category}: {count} ({100 * count / len(test_examples):.1f}%)")


## Cell 8 - TensorFlow Lite conversion & INT8 quantization

Follows the same PyTorch -> TensorFlow -> TFLite steps as Cell 4 in
`receipt_ner_gpu_training.ipynb`, just extended to handle the extra
`pixel_values` input. A couple of things worth knowing, both checked
directly against the real trained model rather than assumed.

Converting from PyTorch to TensorFlow and reloading it gave 100% matching
predictions on a real forward pass, so `from_pt=True` transfers the
weights correctly here too. Full INT8 quantization doesn't work for this
model, though - it fails with a shape mismatch in a TRANSPOSE op inside
the visual patch embedding, an op v1 never had since it has no image
encoder. The fallback code below already handles exactly this case, same
as v1: it drops down to dynamic-range INT8, which only quantizes the
weights, and that produces a working file - just a bigger one, about
146MB versus v1's 100-115MB, since LayoutLMv3-base has more parameters
plus the visual backbone. The plan to deliver it over-the-air rather than
bundling it in the APK (same as v1) still holds, just for a slightly
bigger download.

In [ ]:
from transformers import TFLayoutLMv3ForTokenClassification
import tempfile

tf_model = TFLayoutLMv3ForTokenClassification.from_pretrained(SAVE_DIR, from_pt=True)

# Sanity check (same as v1's Cell 4): cross-framework conversion is usually reliable but worth verifying rather than trusting it blindly - confirmed 100% agreement before this cell was written.
_sample = tokenized_datasets["test"][0]
_device = trainer.model.device
_pt_input_ids = torch.tensor([_sample["input_ids"]]).to(_device)
_pt_attention_mask = torch.tensor([_sample["attention_mask"]]).to(_device)
_pt_bbox = torch.tensor([_sample["bbox"]]).to(_device)
_pt_pixel_values = torch.tensor(np.array([_sample["pixel_values"]]), dtype=torch.float32).to(_device)
with torch.no_grad():
    _pt_pred = trainer.model(
        input_ids=_pt_input_ids, attention_mask=_pt_attention_mask,
        bbox=_pt_bbox, pixel_values=_pt_pixel_values,
    ).logits.argmax(-1)

_tf_input_ids = tf.constant([_sample["input_ids"]], dtype=tf.int32)
_tf_attention_mask = tf.constant([_sample["attention_mask"]], dtype=tf.int32)
_tf_bbox = tf.constant([_sample["bbox"]], dtype=tf.int32)
_tf_pixel_values = tf.constant(np.array([_sample["pixel_values"]]), dtype=tf.float32)
_tf_pred = tf.argmax(
    tf_model(
        input_ids=_tf_input_ids, attention_mask=_tf_attention_mask,
        bbox=_tf_bbox, pixel_values=_tf_pixel_values,
    ).logits, axis=-1,
)

_agreement = (_pt_pred.cpu().numpy().flatten() == np.array(_tf_pred).flatten()).mean()
print(f"PyTorch vs. reloaded-TensorFlow prediction agreement on one sample: {_agreement:.1%}")
if _agreement < 0.99:
    print("WARNING: low agreement - the from_pt=True conversion may not have transferred weights correctly.")


In [ ]:
class InferenceWrapper(tf.Module):
    """Fixed-shape, fixed-batch-size (1) wrapper - on-device inference is
    always one receipt at a time. pixel_values shape [1,3,224,224] matches
    LayoutLMv3ImageProcessor's fixed output size regardless of the source
    image's own dimensions."""

    def __init__(self, hf_model):
        super().__init__()
        self.model = hf_model

    @tf.function(input_signature=[
        tf.TensorSpec(shape=[1, MAX_LENGTH], dtype=tf.int32, name="input_ids"),
        tf.TensorSpec(shape=[1, MAX_LENGTH], dtype=tf.int32, name="attention_mask"),
        tf.TensorSpec(shape=[1, MAX_LENGTH, 4], dtype=tf.int32, name="bbox"),
        tf.TensorSpec(shape=[1, 3, 224, 224], dtype=tf.float32, name="pixel_values"),
    ])
    def serve(self, input_ids, attention_mask, bbox, pixel_values):
        output = self.model(
            input_ids=input_ids, attention_mask=attention_mask,
            bbox=bbox, pixel_values=pixel_values, training=False,
        )
        return {"logits": output.logits}


wrapper = InferenceWrapper(tf_model)
saved_model_dir = tempfile.mkdtemp()
tf.saved_model.save(wrapper, saved_model_dir, signatures={"serving_default": wrapper.serve})
print(f"Saved intermediate TensorFlow SavedModel to {saved_model_dir}")


In [ ]:
# --- Representative dataset for INT8 calibration -------------------------
# Same as v1's Cell 4: training pads dynamically per-batch, but the deployed model needs a fixed [1, MAX_LENGTH] shape. pixel_values is already fixed-size, so it needs no padding.
def _pad_to_max_length(example):
    length = len(example["input_ids"])
    pad_len = MAX_LENGTH - length
    if pad_len >= 0:
        input_ids = example["input_ids"] + [tokenizer.pad_token_id] * pad_len
        attention_mask = example["attention_mask"] + [0] * pad_len
        bbox = example["bbox"] + [[0, 0, 0, 0]] * pad_len
    else:
        input_ids = example["input_ids"][:MAX_LENGTH]
        attention_mask = example["attention_mask"][:MAX_LENGTH]
        bbox = example["bbox"][:MAX_LENGTH]
    return input_ids, attention_mask, bbox


def representative_data_gen():
    sample_count = min(100, len(tokenized_datasets["train"]))
    for i in range(sample_count):
        example = tokenized_datasets["train"][i]
        input_ids, attention_mask, bbox = _pad_to_max_length(example)
        yield [
            np.array([input_ids], dtype=np.int32),
            np.array([attention_mask], dtype=np.int32),
            np.array([bbox], dtype=np.int32),
            np.array([example["pixel_values"]], dtype=np.float32),
        ]


def convert_full_int8(saved_model_dir):
    converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_data_gen
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.float32
    return converter.convert()


def convert_dynamic_range_int8(saved_model_dir):
    converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]  # weights-only int8, no representative dataset needed
    return converter.convert()


try:
    tflite_model = convert_full_int8(saved_model_dir)
    conversion_path = "full INT8 (TFLITE_BUILTINS_INT8, activations + weights quantized)"
except Exception as e:
    # This really does happen for this architecture (the visual patch embedding's TRANSPOSE op has no full-integer kernel) - not a hypothetical fallback.
    print(f"Full INT8 conversion failed ({e}); falling back to dynamic-range INT8 (weights only)...")
    tflite_model = convert_dynamic_range_int8(saved_model_dir)
    conversion_path = "dynamic-range INT8 (weights only - full INT8 was attempted first and failed)"

print(f"\nConversion path used: {conversion_path}")


In [ ]:
OUTPUT_TFLITE = "receipt_parser_v3_quantized.tflite"
OUTPUT_VOCAB = "receipt_parser_v3_vocab.txt"
OUTPUT_LABELS = "receipt_parser_v3_labels.json"

with open(OUTPUT_TFLITE, "wb") as f:
    f.write(tflite_model)

size_mb = len(tflite_model) / (1024 * 1024)
print(f"Saved {OUTPUT_TFLITE}: {size_mb:.1f} MB")
if size_mb > 160:
    print("NOTE: larger than the ~146MB measured during development - not necessarily a problem, "
          "just confirming the OTA delivery plan (not bundled in the APK) still applies.")

# LayoutLMv3 uses byte-level BPE (like RoBERTa), not v1's WordPiece - a Dart-side tokenizer would need both vocab.txt and merges.txt, not just vocab.txt.
vocab_by_id = sorted(tokenizer.get_vocab().items(), key=lambda kv: kv[1])
with open(OUTPUT_VOCAB, "w", encoding="utf-8", newline="\n") as f:
    f.write("\n".join(token for token, _ in vocab_by_id))

label_config = {
    "labels": LABELS,
    "fields": FIELDS,
    "max_length": MAX_LENGTH,
    "cls_token_id": tokenizer.cls_token_id,
    "sep_token_id": tokenizer.sep_token_id,
    "pad_token_id": tokenizer.pad_token_id,
    "unk_token": tokenizer.unk_token,
    "bbox_scale_max": 1000,
    # v3-specific: on-device inference also needs to resize/normalize the receipt image to this size and feed it as pixel_values - new requirement v1 never had.
    "image_size": [224, 224],
}
with open(OUTPUT_LABELS, "w", encoding="utf-8") as f:
    json.dump(label_config, f, indent=2)

print(f"Saved {OUTPUT_VOCAB} and {OUTPUT_LABELS}")
print("\nDownload all files from the file browser before the runtime resets.")
